In [7]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_percentage_error
import matplotlib.pyplot as plt

# -------------------------------
# 1) Load the CSV Data
# -------------------------------
csv_filename = "APO_with_sentiment.csv"
df = pd.read_csv(csv_filename)

print("Original columns:", df.columns.tolist())

# Convert Date column and set as index
if 'Date' in df.columns:
    df['date'] = pd.to_datetime(df['Date'], errors='coerce')
    df = df.sort_values('date').reset_index(drop=True)
    df.set_index('date', inplace=True)
    
# Rename columns to match our expectations
df = df.rename(columns={
    'Close_Prices': 'Close',
    'High_Prices': 'High',
    'Low_Prices': 'Low',
    'Open_Prices': 'Open'
})

# Convert numeric columns to proper numeric types
numeric_cols = ['Close', 'High', 'Low', 'Open', 'Volume']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop unneeded columns if they exist
cols_to_drop = ['Symbol', 'sentiment', 'Date']
for col in cols_to_drop:
    if col in df.columns:
        df = df.drop(columns=[col])

# If Volume is all NaN, drop it
if 'Volume' in df.columns and df['Volume'].isna().all():
    print("Volume column is all NaN. Dropping it.")
    df = df.drop(columns=['Volume'])

print("Data after renaming and type conversion:", df.shape)
print(df.head())

# -------------------------------
# 2) Compute Technical Indicators
# -------------------------------
def compute_indicators(df):
    # RSI (14-day)
    window_rsi = 14
    delta = df['Close'].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    roll_up = gain.ewm(span=window_rsi).mean()
    roll_down = loss.ewm(span=window_rsi).mean()
    rs = roll_up / roll_down
    df['RSI'] = 100.0 - (100.0 / (1.0 + rs))
    
    # MACD (12,26,9)
    ema12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema26 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = ema12 - ema26
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_Hist'] = df['MACD'] - df['MACD_Signal']
    
    # Bollinger Bands (20-day SMA)
    window_bb = 20
    sma = df['Close'].rolling(window=window_bb).mean()
    std = df['Close'].rolling(window=window_bb).std()
    df['BB_Mid'] = sma
    df['BB_Upper'] = sma + 2 * std
    df['BB_Lower'] = sma - 2 * std
    
    # Print missing values before filling/dropping
    print("Missing values before filling/dropping:\n", df.isna().sum())
    
    # Fill missing values using forward and backward fill, then drop any remaining NaNs
    df = df.ffill().bfill().dropna()
    return df

df = compute_indicators(df)
print("Data shape after computing indicators:", df.shape)
print(df.head())

# -------------------------------
# 3) Prepare Data for LSTM (Windowed Sequences)
# -------------------------------
def prepare_data(df, feature_cols, target_col='Close', window_size=60):
    data = df[feature_cols].values  # shape: (num_samples, num_features)
    target = df[target_col].values.reshape(-1, 1)
    
    # Combine features and target for a single scaler transform
    scaler = MinMaxScaler(feature_range=(0, 1))
    data_combined = np.hstack([data, target])
    data_scaled = scaler.fit_transform(data_combined)
    
    n_features = len(feature_cols)
    features_scaled = data_scaled[:, :n_features]
    target_scaled = data_scaled[:, n_features]
    
    X, y = [], []
    for i in range(window_size, len(df)):
        X.append(features_scaled[i - window_size:i])
        y.append(target_scaled[i])
    X = np.array(X)  # shape: (samples, window_size, n_features)
    y = np.array(y)  # shape: (samples,)
    return X, y, scaler

feature_cols = ['Close', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Mid', 'BB_Upper', 'BB_Lower']
WINDOW_SIZE = 60
X, y, scaler = prepare_data(df, feature_cols=feature_cols, target_col='Close', window_size=WINDOW_SIZE)
print("Prepared X shape:", X.shape)
print("Prepared y shape:", y.shape)

# -------------------------------
# 4) Custom PyTorch Dataset
# -------------------------------
class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# -------------------------------
# 5) Define the LSTM Model
# -------------------------------
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=50, num_layers=2, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        batch_size = x.size(0)
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = out[:, -1, :]  # take the output of the last time step
        out = self.fc(out)
        return out

# -------------------------------
# 6) Training Loop with Gradient Clipping and Early Stopping
# -------------------------------
class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-5):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = None
        self.counter = 0
        self.should_stop = False
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif self.best_loss - val_loss > self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

def train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=50, early_stopping=None, clip_value=1.0):
    model.to(device)
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.float().to(device)
            y_batch = y_batch.float().to(device)
            optimizer.zero_grad()
            outputs = model(X_batch).squeeze()
            loss = criterion(outputs, y_batch)
            loss.backward()
            # Apply gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_value)
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
        train_loss /= len(train_loader.dataset)
    
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.float().to(device)
                y_batch = y_batch.float().to(device)
                outputs = model(X_batch).squeeze()
                loss = criterion(outputs, y_batch)
                val_loss += loss.item() * X_batch.size(0)
        val_loss /= len(val_loader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")
    
        if early_stopping is not None:
            early_stopping(val_loss)
            if early_stopping.should_stop:
                print("Early stopping triggered!")
                break

# -------------------------------
# 7) Walk-Forward Cross Validation
# -------------------------------
def walk_forward_cv(X, y, scaler, n_splits=3, batch_size=32, hidden_dim=100,
                    num_layers=2, dropout=0.2, lr=1e-4, epochs=50, patience=15):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    total_size = len(X)
    fold_size = total_size // n_splits
    start = 0

    for fold in range(n_splits):
        end = start + fold_size
        train_idx = range(0, end)
        val_idx = range(end, min(end + fold_size, total_size))
        if len(val_idx) == 0:
            print(f"Skipping fold {fold+1}: no validation data.")
            break

        print(f"\n===== Fold {fold+1} =====")
        print(f"Train indices: {0} to {end-1}, Validation indices: {end} to {min(end + fold_size, total_size)-1}")

        X_train, y_train = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]

        train_loader = DataLoader(StockDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(StockDataset(X_val, y_val), batch_size=batch_size, shuffle=False)

        model = LSTMModel(input_dim=X.shape[2], hidden_dim=hidden_dim, num_layers=num_layers, dropout=dropout)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        early_stopper = EarlyStopping(patience=patience)

        train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=epochs,
                    early_stopping=early_stopper, clip_value=1.0)
        start = end

# Hyperparameters for CV and final training
N_SPLITS = 3
BATCH_SIZE = 32
EPOCHS = 50
LR = 1e-4  # Lower learning rate for stability
HIDDEN_DIM = 100
NUM_LAYERS = 2
DROPOUT = 0.2
PATIENCE = 15

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

print("\n--- Walk-Forward Cross Validation ---")
walk_forward_cv(X, y, scaler=scaler, n_splits=N_SPLITS, batch_size=BATCH_SIZE,
                hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT,
                lr=LR, epochs=EPOCHS, patience=PATIENCE)

# -------------------------------
# 8) Final Model Training & Future Prediction
# -------------------------------
FINAL_HOLDOUT_SIZE = 100
if len(X) <= FINAL_HOLDOUT_SIZE:
    print("Not enough data for a final holdout. Skipping final test step.")
else:
    split_point = len(X) - FINAL_HOLDOUT_SIZE
    X_train_all, y_train_all = X[:split_point], y[:split_point]
    X_test_final, y_test_final = X[split_point:], y[split_point:]

    print(f"\nTraining final model on {len(X_train_all)} samples, testing on {len(X_test_final)} samples.")
    train_loader = DataLoader(StockDataset(X_train_all, y_train_all), batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(StockDataset(X_test_final, y_test_final), batch_size=BATCH_SIZE, shuffle=False)

    model_final = LSTMModel(input_dim=X.shape[2], hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT)
    model_final.to(device)
    criterion_final = nn.MSELoss()
    optimizer_final = optim.Adam(model_final.parameters(), lr=LR)
    early_stopper_final = EarlyStopping(patience=PATIENCE)

    print("\n---- Training Final Model ----")
    for epoch in range(EPOCHS):
        model_final.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.float().to(device)
            y_batch = y_batch.float().to(device)
            optimizer_final.zero_grad()
            outputs = model_final(X_batch).squeeze()
            loss = criterion_final(outputs, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_final.parameters(), 1.0)
            optimizer_final.step()
            train_loss += loss.item() * X_batch.size(0)
        train_loss /= len(train_loader.dataset)

        model_final.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.float().to(device)
                y_batch = y_batch.float().to(device)
                outputs = model_final(X_batch).squeeze()
                loss = criterion_final(outputs, y_batch)
                val_loss += loss.item() * X_batch.size(0)
        val_loss /= len(test_loader.dataset)
        print(f"Epoch [{epoch+1}/{EPOCHS}], Train Loss: {train_loss:.6f}, Test Loss (scaled): {val_loss:.6f}")
        early_stopper_final(val_loss)
        if early_stopper_final.should_stop:
            print("Early stopping triggered!")
            break

    # Generate predictions on final holdout set
    model_final.eval()
    preds_list = []
    with torch.no_grad():
        for X_batch, _ in test_loader:
            X_batch = X_batch.float().to(device)
            preds = model_final(X_batch).squeeze()
            preds_list.append(preds.cpu().numpy())
    preds_np = np.concatenate(preds_list).ravel()

    # Invert scaling: we combined features and target earlier.
    # Create an array of zeros for the feature columns to append the predicted target.
    n_features = len(feature_cols)
    preds_combined = np.hstack([np.zeros((len(preds_np), n_features)), preds_np.reshape(-1, 1)])
    unscaled_preds = scaler.inverse_transform(preds_combined)[:, -1]

    y_test_combined = np.hstack([np.zeros((len(y_test_final), n_features)), y_test_final.reshape(-1, 1)])
    unscaled_y_test = scaler.inverse_transform(y_test_combined)[:, -1]

    # Check that no NaNs exist before computing metrics
    if np.isnan(unscaled_preds).any() or np.isnan(unscaled_y_test).any():
        raise ValueError("Predictions or true values contain NaN values.")

    rmse_final = np.sqrt(np.mean((unscaled_preds - unscaled_y_test)**2))
    r2_final = r2_score(unscaled_y_test, unscaled_preds)
    mape_final = mean_absolute_percentage_error(unscaled_y_test, unscaled_preds) * 100

    print(f"\nFinal Holdout RMSE: {rmse_final:.4f}")
    print(f"Final Holdout R^2: {r2_final:.4f}")
    print(f"Final Holdout MAPE: {mape_final:.2f}%")

    # Plot the last 40 days of actual vs. predicted prices
    comparison_df = pd.DataFrame({
        "Actual": unscaled_y_test,
        "Predicted": unscaled_preds
    })
    print("\nFinal Holdout Comparison (last 40):")
    print(comparison_df.tail(40))

    plt.figure(figsize=(10, 5))
    plt.plot(comparison_df.index[-40:], comparison_df["Actual"].tail(40), label="Actual")
    plt.plot(comparison_df.index[-40:], comparison_df["Predicted"].tail(40), label="Predicted")
    plt.title("Final Holdout: Actual vs. Predicted Prices (Last 40 Days)")
    plt.xlabel("Time Step")
    plt.ylabel("Price")
    plt.legend()
    plt.show()


Original columns: ['Date', 'Close_Prices', 'High_Prices', 'Low_Prices', 'Open_Prices', 'Volume', 'Symbol', 'sentiment']
Volume column is all NaN. Dropping it.
Data after renaming and type conversion: (131, 4)
                                    Close        High         Low     Open
date                                                                      
1970-01-01 00:00:00.000000145  148.899994  141.199997  148.000000  5772700
1970-01-01 00:00:00.000000147  152.660004  146.440002  151.910004  4379300
1970-01-01 00:00:00.000000147  152.660004  146.440002  151.910004  4379300
1970-01-01 00:00:00.000000147  152.660004  146.440002  151.910004  4379300
1970-01-01 00:00:00.000000147  152.660004  146.440002  151.910004  4379300
Missing values before filling/dropping:
 Close           0
High            0
Low             0
Open            0
RSI             1
MACD            0
MACD_Signal     0
MACD_Hist       0
BB_Mid         19
BB_Upper       19
BB_Lower       19
dtype: int64
Data shape aft